In [1]:
from pathlib import Path
import re
from gensim.models import Word2Vec
from gensim.models.phrases import Phrases, Phraser
import spacy

nlp = spacy.load("es_core_news_sm", disable=["ner", "parser"])

/home/juancho/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [19]:
def lemmatize(texto):
    doc = nlp(texto)
    return [token.text.lower() for token in doc if token.is_alpha and not token.is_stop]

In [7]:
directorio_con_datos = "../buscador/all_pages_clean/"
ruta = Path(directorio_con_datos)

print("Cargando documentos...")
archivos = list(ruta.glob("*.txt"))

def extraer_numero(archivo):
    numeros = re.findall(r'(\d+)', archivo.stem)
    return int(numeros[0]) if numeros else 0

archivos.sort(key=extraer_numero)

documentos = []
nombres_archivos = []
for archivo in archivos:
    try:
        with open(archivo, 'r', encoding='utf-8') as f:
            contenido = f.read()
            if contenido.strip():
                documentos.append(contenido)
                nombres_archivos.append(archivo.name)
    except Exception as e:
        print(f"Error leyendo {archivo.name}: {e}")

print(f"✅ Cargados {len(documentos)} documentos")


Cargando documentos...
✅ Cargados 548 documentos


In [ ]:
corpus_lemmatized = [lemmatize(doc) for doc in documentos]
min_count_bigram = 10  # Si tu corpus es pequeño, pon 1 para no descartar ningún par
threshold_bigram = 30

bigram_model = Phrases(
    sentences=corpus_lemmatized, 
    min_count=min_count_bigram,   # Ignora pares que aparecen menos de X veces
    threshold=threshold_bigram    # Controla la fuerza de la asociación
)

# Convierte el modelo en un objeto "Phraser" para aplicar más rápido
bigram_phraser = Phraser(bigram_model)

# --- 3. APLICAR BIGRAMAS AL CORPUS ---
# Ahora convertimos cada documento: donde haya un bigrama, lo fusionamos con "_"
corpus_con_bigramas = [bigram_phraser[doc] for doc in corpus_lemmatized]

print(corpus_con_bigramas)

In [ ]:
modelo = Word2Vec(
    sentences=corpus_con_bigramas,
    vector_size=150,
    window=7,
    min_count=5,
    workers=4,
    sg=1,
    epochs=25,
    sample=0,
    negative=10
)


similares = modelo.wv.most_similar("corrupción", topn=400)
print(similares)

[('clasífico', 0.5617634654045105), ('puntuándolo', 0.5508641004562378), ('obstáculos', 0.537205696105957), ('ceesp', 0.507417619228363), ('toleranciar', 0.5003739595413208), ('esfuerzos', 0.4972556233406067), ('transcript_generated', 0.4910920560359955), ('posición_ranking', 0.4840151369571686), ('desproporcionadamente', 0.48326805233955383), ('coeficientes', 0.481080561876297), ('práctica_corrupto', 0.47821521759033203), ('estadística_detallado', 0.4738250970840454), ('dato_complet', 0.47163644433021545), ('for_free', 0.46839526295661926), ('violación_des', 0.468344122171402), ('haberno', 0.46818801760673523), ('alarmant', 0.46652838587760925), ('transversalizar', 0.4663521647453308), ('moquegua', 0.4646938145160675), ('leis', 0.4623608887195587)]


In [ ]:
similares2 = similares

In [ ]:
with open("../datos/word2vec_similares.txt", 'w', encoding='utf-8') as f:
    for palabra, similitud in similares:
        f.write(f"{palabra}: {similitud}\n")

# Construir matriz de documentos

In [ ]:
import numpy as np
import pandas as pd

# Obtén todas las palabras y sus vectores
palabras = list(modelo.wv.key_to_index.keys())  # Lista de palabras
vectores = np.array([modelo.wv[palabra] for palabra in palabras])  # Matriz de vectores

# Crea un DataFrame (opcional)
df_word2vec_vocab = pd.DataFrame(
    vectores, 
    index=palabras, 
    columns=[f"dim_{i}" for i in range(vectores.shape[1])]
)

print(f"Forma: {vectores.shape}")  # (n_palabras, 100)
print(df_word2vec_vocab.head())

# Guardar a CSV
df_word2vec_vocab.to_csv("matriz_word2vec_vocabulario.csv", encoding='utf-8')

TF-IDF

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Primero, crea la matriz TF-IDF (ya la tienes con tu código)
# docs_txt = [' '.join(doc) for doc in corpus_final]
# matriz_tfidf = tfidf_vectorizer.fit_transform(docs_txt)

# Ahora, crea una matriz de Word2Vec ponderada por TF-IDF
# Esto NO es una matriz densa normal, sino una lista de matrices por documento.
# Lo que hacemos es: para cada documento, creamos una fila con los vectores 
# de las palabras multiplicados por su peso TF-IDF.

# Opción simple: para cada documento, un vector promedio ponderado
from scipy.sparse import csr_matrix

def obtener_vector_documento_tfidf(tokens, modelo, vectorizer, docs_txt, idx_doc):
    """
    Calcula el vector del documento como la suma ponderada de los vectores
    de cada palabra por su peso TF-IDF.
    """
    # Obtén los pesos TF-IDF para este documento
    # (es una fila de la matriz dispersa)
    fila_tfidf = matriz_tfidf[idx_doc]
    
    # Obtén las palabras del vocabulario TF-IDF
    palabras_tfidf = vectorizer.get_feature_names_out()
    
    # Vector acumulador
    vector_acumulado = np.zeros(modelo.vector_size)
    
    # Itera sobre las palabras que tienen peso TF-IDF > 0 en este documento
    for j, peso in zip(fila_tfidf.indices, fila_tfidf.data):
        palabra = palabras_tfidf[j]
        if palabra in modelo.wv:
            vector_acumulado += peso * modelo.wv[palabra]
    
    return vector_acumulado

# Calcula los vectores ponderados para todos los documentos
vectores_tfidf_weighted = np.array([
    obtener_vector_documento_tfidf(doc, modelo, tfidf_vectorizer, docs_txt, i)
    for i, doc in enumerate(corpus_final)
])

print(f"Forma: {vectores_tfidf_weighted.shape}")  # (500, 100)

# Guardar
df_tfidf_weighted = pd.DataFrame(
    vectores_tfidf_weighted,
    columns=[f"dim_{i}" for i in range(vectores_tfidf_weighted.shape[1])]
)
df_tfidf_weighted.to_csv("matriz_word2vec_tfidf_documentos.csv", encoding='utf-8')

NameError: name 'np' is not defined